# Factory Design Pattern 

explained using the classic Logistics (Transport) example.

#### The Concept
The Factory Pattern is a creational pattern that provides a way to create objects without specifying their exact class. 

**The Problem**: If you have code that says `new Truck()`, `new Truck()`, `new Truck()` everywhere, and suddenly you need to change it to `new ElectricTruck()`, you have to edit 100 files. 

**The Solution**: You call a special method `create_transport("road")`. The Factory decides whether to give you a `Truck` or an `ElectricTruck`.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we use an Interface (or `Abstract Class`) for the products. The Factory is often a separate class containing a method with a large `if/else` or `switch` statement to decide which object to instantiate.

#### THE INTERFACE (Abstract Product)

In [2]:
from abc import ABC, abstractmethod

class Transport(ABC):
    @abstractmethod
    def deliver(self) -> str:
        """Abstract method ensuring all products behave the same way."""
        pass

#### THE CONCRETE PRODUCTS

In [10]:
class Truck(Transport):
    def deliver(self):
        return ("🚚 Delivering by land in a box.")

class Ship(Transport):
    def deliver(self):
        return ("🚢 Delivering by sea in a container.")

class Drone(Transport):
    def deliver(self):
        return ("🚁 Delivering by air (small package).")

#### THE FACTORY

In [11]:
class LogisticsFactory:
    
    @staticmethod
    def get_transport(transport_type: str) -> Transport:
        """
        Factory Method.
        
        param transport_type: "road", "sea", or "air"
        returns: A Concrete Transport Object
        """
        # Using the modern 'match' statement (Python 3.10+)
        match transport_type.lower():
            case "road":
                return Truck()
            case "sea":
                return Ship()
            case "air":
                return Drone()
            case _:
                raise ValueError(f"Unknown transport type: {transport_type}")

    __TRANSPORTS = {
        "road": Truck(),
        "sea": Ship(),
        "air": Drone()
    }
    
    @staticmethod
    def get_transport_v2(transport_type: str) -> Transport:
        """
        Factory Method.
        
        param transport_type: "road", "sea", or "air"
        returns: A Concrete Transport Object
        """
        transport = LogisticsFactory.__TRANSPORTS.get(transport_type)
        if not transport:
            raise ValueError(f"Unknown transport type: {transport_type}")
        return transport
        

#### CLIENT CODE

In [12]:
def main():
    factory = LogisticsFactory()

    # Define a list of requests to process
    requests: list[str] = ["road", "sea", "air", "space"]

    print("--- Logistics App Started ---\n")

    for req in requests:
        try:
            # The client doesn't know (or care) which class is created,
            # it just knows it gets a 'Transport' object.
            vehicle: Transport = factory.get_transport(req)
            print(f"Request: {req.ljust(5)} | {vehicle.deliver()}")
        
        except ValueError as e:
            print(f"Request: {req.ljust(5)} | Error: {e}")

if __name__ == "__main__":
    main()

--- Logistics App Started ---

Request: road  | 🚚 Delivering by land in a box.
Request: sea   | 🚢 Delivering by sea in a container.
Request: air   | 🚁 Delivering by air (small package).
Request: space | Error: Unknown transport type: space


## The Pythonic Way (Dynamic Registry)

In Python, **Classes are First-Class Objects**. We can pass classes around just like variables. Instead of a giant `if/else` block inside a Factory class, we can simply map strings to the Classes in a `Dictionary`. This is faster, cleaner, and easier to extend.

#### THE PRODUCTS (Just Classes)

In [34]:
class Truck:
    def deliver(self): print("🚚 Truck Delivery")

class Ship:
    def deliver(self): print("🚢 Ship Delivery")

class Drone:
    def deliver(self): print("🚁 Drone Delivery")

#### THE PYTHONIC FACTORY

We map the "ID" directly to the "Class Reference"

In [36]:
transport_registry = {
    "road": Truck,
    "sea":  Ship,
    "air":  Drone
}

def get_transport(kind: str):
    """
    A simple function acts as the Factory.
    """
    try:
        # 1. Get the class from the dictionary
        transport_class = transport_registry[kind]
        
        # 2. Instantiate it () and return
        return transport_class()
        
    except KeyError:
        raise ValueError(f"Unknown transport type: {kind}")

#### CLIENT CODE

In [37]:
def main():
    print("--- Pythonic Factory ---")
    
    # 1. Usage
    my_vehicle = get_transport("air")
    my_vehicle.deliver()

    # 2. Dynamic Extension? 
    # In Java, adding a "Train" requires editing the Factory class (OCP violation).
    # In Python, we can just add to the dictionary at runtime!
    
    class Train:
        def deliver(self): print("🚂 Train Delivery")
        
    transport_registry["rail"] = Train
    
    my_train = get_transport("rail")
    my_train.deliver()

if __name__ == "__main__":
    main()

--- Pythonic Factory ---
🚁 Drone Delivery
🚂 Train Delivery


#### Key Differences

| Feature             | Classic OOP                                                   | Pythonic                                                      |
|---------------------|---------------------------------------------------------------|----------------------------------------------------------------|
| **Logic**           | `if / elif / else` blocks inside a Factory class.             | Dictionary lookup (`key → class`).                             |
| **Extensibility**   | Hard — must modify the factory code to add new types.         | Easy — add a new entry to the dictionary (even at runtime).    |
| **Class Reference** | Hardcoded instantiation calls (e.g., `new Truck()`).          | Class stored as a dictionary value and instantiated dynamically. |


#### When to use which?

- **Java Way**: When the creation logic is complex (e.g., `Truck` needs 5 arguments from a config file, `Ship` needs 2 arguments from a database). The `if/else` block allows custom setup for each.
- **Pythonic Way**: When the creation logic is uniform (every class takes similar arguments) and you want a clean, table-driven approach.

## More Pythonic Way

In Python, **Classes are First-Class Objects**. We can pass classes around just like variables. Instead of a giant if/else block inside a Factory class, we can simply map strings to the Classes in a **Dictionary**. This is faster, cleaner, and easier to extend.


#### Why this is "More Pythonic"

- Functions over Classes: We don't need a `LogisticsFactory` class. A simple function `get_transport()` is sufficient. Python functions are first-class objects.
- `Protocols` over `ABC`s: We use Protocol (Duck Typing). The classes `Truck` and `Ship` do not need to inherit from `Transport`. As long as they have the deliver method, they work.
- Dictionary Registry: instead of a long `if/elif` or `match/case chain`, we use a dictionary to map strings to Classes. This is faster and cleaner.

#### THE PROTOCOL (Interface)

We use a `Protocol` to define the shape of the object. `runtime_checkable` allows us to use `isinstance()` if needed.

In [14]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Transport(Protocol):
    def deliver(self) -> str: ...

#### THE CONCRETE PRODUCTS

Notice: NO inheritance from `Transport`. These are simple, standalone classes.

In [15]:
class Truck:
    def deliver(self) -> str:
        return "Truck: Delivering by land in a box."

class Ship:
    def deliver(self) -> str:
        return "Ship: Delivering by sea in a container."

class Drone:
    def deliver(self) -> str:
        return "Drone: Delivering by air."

#### THE PYTHONIC FACTORY (Function + Registry)

A `dictionary` mapping keys to the Class Types (not instances) `[Transport]` means "Any class that implements the Transport protocol"

In [17]:
from typing import Type

TRANSPORT_REGISTRY: dict[str, Type[Transport]] = {
    "road": Truck,
    "sea":  Ship,
    "air":  Drone,
}

def get_transport(mode: str) -> Transport:
    """
    The Factory Function.
    Looks up the class in the registry and instantiates it.
    """
    try:
        # 1. Fetch the class (e.g., Truck)
        vehicle_class = TRANSPORT_REGISTRY[mode.lower()]
        
        # 2. Instantiate it (e.g., Truck())
        return vehicle_class()
        
    except KeyError:
        raise ValueError(f"Unknown transport mode: {mode}")

#### CLIENT CODE

In [18]:
def main():
    print("--- Pythonic Factory App ---")

    # List of requested modes
    requests = ["road", "sea", "air", "space"]

    for mode in requests:
        try:
            # The client simply calls the function
            vehicle = get_transport(mode)
            print(f"Mode '{mode}': {vehicle.deliver()}")
        
        except ValueError as e:
            print(f"Mode '{mode}': Error -> {e}")

if __name__ == "__main__":
    main()

--- Pythonic Factory App ---
Mode 'road': Truck: Delivering by land in a box.
Mode 'sea': Ship: Delivering by sea in a container.
Mode 'air': Drone: Delivering by air.
Mode 'space': Error -> Unknown transport mode: space


#### Key Differences

| Feature            | Classic OOP                                                   | Pythonic                                                      |
|--------------------|----------------------------------------------------------------|----------------------------------------------------------------|
| **Logic**          | `if / elif / else` blocks inside a Factory class.              | Dictionary lookup (`key → class`).                             |
| **Extensibility**  | Hard — must modify factory code to add new types.              | Easy — add a new entry to the dictionary (even at runtime).    |
| **Class Reference**| Hardcoded instantiation (`new Truck()` style calls).           | Class stored as a dict value and instantiated dynamically.    |


#### When to use which?

- **Java Way**: When the creation logic is complex (e.g., `Truck` needs 5 arguments from a config file, `Ship` needs 2 arguments from a database). The `if/else` block allows custom setup for each.
- **Pythonic Way**: When the creation logic is uniform (every class takes similar arguments) and you want a clean, table-driven approach.

# Factory Design Pattern 

explained using a common enterprise software scenario: Multi-Cloud Storage Client.

#### The Scenario: Cloud Agnostic File Uploads

Imagine you are building a backend system (like Dropbox or a Backup Service). You want your users to choose where they store their files: **AWS S3, Azure Blob Storage**, or **Local Disk** (for development).

- **The Problem**: AWS requires an `access_key` and `region`. Azure requires a `connection_string`. Local disk requires a `root_path`.
- **The Goal**: The rest of your application shouldn't care about these details. It just wants to call **storage.upload(file)**.

## The Classic OOP Way (Java-Style)

We define a strict interface `IStorageService`. We implement concrete classes for each provider. The `StorageFactory` handles the complex initialization logic (parsing specific configs for specific providers), isolating it from the client code.

#### THE PRODUCT INTERFACE

In [19]:
from abc import ABC, abstractmethod

class IStorageService(ABC):
    @abstractmethod
    def upload(self, file_name: str, content: bytes):
        pass

    @abstractmethod
    def get_url(self, file_name: str) -> str:
        pass

#### CONCRETE PRODUCTS (Providers)

In [20]:
class S3StorageService(IStorageService):
    def __init__(self, api_key: str, region: str):
        self.api_key = api_key
        self.region = region
        print(f"🔌 Connecting to AWS S3 ({region})...")

    def upload(self, file_name: str, content: bytes):
        print(f"☁️ [AWS] Uploading '{file_name}' to S3 Bucket")

    def get_url(self, file_name: str) -> str:
        return f"https://s3.{self.region}.amazonaws.com/{file_name}"

class AzureBlobService(IStorageService):
    def __init__(self, connection_string: str):
        self.conn_str = connection_string
        print(f"🔌 Connecting to Azure Blob...")

    def upload(self, file_name: str, content: bytes):
        print(f"☁️ [Azure] Uploading '{file_name}' to Blob Container")

    def get_url(self, file_name: str) -> str:
        return f"https://azure.microsoft.com/blob/{file_name}"

class LocalDiskService(IStorageService):
    def __init__(self, root_path: str):
        self.root_path = root_path
        print(f"🔌 Mounting Local Disk at {root_path}...")

    def upload(self, file_name: str, content: bytes):
        print(f"💾 [Disk] Saving '{file_name}' to {self.root_path}")

    def get_url(self, file_name: str) -> str:
        return f"file://{self.root_path}/{file_name}"

#### THE FACTORY (Complex Init Logic)

In [22]:
from typing import Dict

class StorageFactory:
    @staticmethod
    def create_service(provider_type: str, config: Dict[str, str]) -> IStorageService:
        """
        The Factory encapsulates the complexity of validating 
        and extracting the correct config keys for each provider.
        """
        if provider_type == "aws":
            if "key" not in config or "region" not in config:
                raise ValueError("AWS config missing 'key' or 'region'")
            return S3StorageService(config["key"], config["region"])
        
        elif provider_type == "azure":
            if "conn_str" not in config:
                raise ValueError("Azure config missing 'conn_str'")
            return AzureBlobService(config["conn_str"])
            
        elif provider_type == "local":
            return LocalDiskService(config.get("path", "/tmp"))
            
        else:
            raise ValueError(f"Unknown provider: {provider_type}")

#### CLIENT CODE

In [23]:
def main():
    # Imagine this config comes from a YAML file or Environment Variables
    app_config = {
        "provider": "aws",
        "aws_key": "AKIA_TEST_123",
        "aws_region": "us-east-1"
    }

    # 1. Prepare config for factory
    provider = app_config["provider"]
    creds = {"key": app_config["aws_key"], "region": app_config["aws_region"]}

    # 2. Get the service
    storage = StorageFactory.create_service(provider, creds)
    
    # 3. Use it (Polymorphism)
    storage.upload("report.pdf", b"data")
    print(storage.get_url("report.pdf"))

if __name__ == "__main__":
    main()

🔌 Connecting to AWS S3 (us-east-1)...
☁️ [AWS] Uploading 'report.pdf' to S3 Bucket
https://s3.us-east-1.amazonaws.com/report.pdf


## The Pythonic Way (Registration Decorators)

In complex Python frameworks (like Airflow, Django, PyTest), we avoid large `if/elif` blocks in the factory. Instead, we use a Registration Pattern.

- We define a `register_provider` decorator.
- Each service registers itself with a key.
- The factory simply instantiates whatever class corresponds to the key using `**kwargs`.

This makes the system **Open/Closed**: You can add a `GoogleCloudService` in a new file, and it will work without modifying the Factory code.

#### THE REGISTRY (The "Factory" Memory)

In [31]:
from typing import Dict, Any

class ServiceRegistry:
    _creators: Dict[str, Any] = {}

    @classmethod
    def register(cls, provider_type: str):
        """Decorator to register a new storage provider."""
        def decorator(class_ref):
            cls._creators[provider_type] = class_ref
            return class_ref
        return decorator

    @classmethod
    def get_service(cls, provider_type: str, **kwargs):
        """Factory Method"""
        if provider_type not in cls._creators:
            raise ValueError(f"Provider '{provider_type}' not registered")
        
        # Instantiate the class dynamically with whatever args match
        class_ref = cls._creators[provider_type]
        return class_ref(**kwargs)

#### THE PRODUCTS (Self-Registering)

In [32]:
# Example of a Protocol (Duck Typing interface)
class StorageProtocol(Protocol):
    def upload(self, file_name: str): ...

@ServiceRegistry.register("aws")
class S3Service:
    def __init__(self, key: str, region: str = "us-east-1"):
        print(f"🔌 AWS Init: {region}")

    def upload(self, file_name: str):
        print(f"☁️ [AWS] Uploaded {file_name}")

@ServiceRegistry.register("gcp")
class GoogleCloudService:
    def __init__(self, json_key_path: str, bucket: str):
        print(f"🔌 GCP Init: {bucket}")

    def upload(self, file_name: str):
        print(f"☁️ [GCP] Uploaded {file_name}")

@ServiceRegistry.register("local")
class LocalService:
    def __init__(self, path: str = "/tmp"):
        print(f"🔌 Local Init: {path}")

    def upload(self, file_name: str):
        print(f"💾 [Local] Saved {file_name}")

#### CLIENT CODE (Configuration Driven)

In [33]:
def main():
    print("--- Dynamic Pythonic Factory ---")
    
    # Imagine these configs are loaded from 3 different env files
    configs = [
        {"type": "aws", "key": "123", "region": "eu-west-1"},
        {"type": "gcp", "json_key_path": "/etc/key.json", "bucket": "my-bucket"},
        {"type": "local", "path": "/var/www/uploads"}
    ]

    for conf in configs:
        # Extract type, pass the rest as kwargs
        p_type = conf.pop("type")
        
        try:
            # MAGICAL LINE:
            # The factory automatically matches dict keys to constructor arguments
            service = ServiceRegistry.get_service(p_type, **conf)
            service.upload("backup.zip")
            print("-" * 20)
        except TypeError as e:
            print(f"❌ Configuration Error for {p_type}: {e}")

if __name__ == "__main__":
    main()

--- Dynamic Pythonic Factory ---
🔌 AWS Init: eu-west-1
☁️ [AWS] Uploaded backup.zip
--------------------
🔌 GCP Init: my-bucket
☁️ [GCP] Uploaded backup.zip
--------------------
🔌 Local Init: /var/www/uploads
💾 [Local] Saved backup.zip
--------------------


#### Why the Pythonic version is "Industry Standard"

- **Decentralized**: In the OOP version, `StorageFactory` knew about every class. In the Pythonic version, the `S3Service` registers itself. The Factory is just a dumb dictionary wrapper.
- `**kwargs` Injection: Notice how we passed the config dictionary directly into `get_service`. Python automatically unpacks `conf` and matches `provider_type="123"` to the `S3Service` constructor. In Java, you would need a tedious "Builder Pattern" or manual mapping object-to-object.
- **Plugin Friendly**: If you write a library, users can define their own `CustomStorage`, decorate it with `@register("custom")`, and your factory will accept it instantly without you changing your library code.